# 04 - External-predictor tabular models

This notebook introduces **external-predictor tabular models**: classical
regression/tree models trained on engineered daily feature rows built from
calendar information and external satellite/reanalysis/meteorological
predictors, rather than from the target series' own recent history.

It covers:

1. what external-predictor tabular models are and how they differ from the
   Model 0 baselines in `03_baselines.ipynb`;
2. why temporal awareness has to be hand-engineered into these features
   (lags, rolling summaries, anomalies, availability flags) since a plain
   tabular row has no built-in notion of sequence;
3. how the curated external predictor table
   (`data_public/chlorophyll/chlorophyll_predictor_features_curated.csv`)
   is structured and how to load it;
4. a real finding from this project: external predictors alone did not
   clearly beat simple interpolation in this low-data, local setting;
5. a small, runnable example training a tree model on artificial gaps.

## 1. What is an external-predictor tabular model?

A tabular model treats each calendar day as one row of a feature table and
predicts a single target value (here, daily mean chlorophyll-a) from the
other columns in that row -- the same setup as any standard regression or
classification problem. The "external-predictor" qualifier means the
feature columns are restricted to information that does not depend on the
target series' own recent observed values: calendar position, satellite
sea-surface temperature, a satellite chlorophyll proxy, wind, and
upwelling-related variables from a nearby meteorological station and
reanalysis products.

This is deliberately different from a *gap-edge* model (see
`05_gap_edge_residual_models.ipynb`), which is allowed to look at the
target's own value immediately before and after a gap. External-predictor
models are safe to use on **any** gap, including very long ones or gaps
near the edge of the record, because they never require a recent target
observation to exist.

## 2. Why temporal awareness must be engineered by hand

A single row of a tabular model has no built-in concept of "yesterday" or
"a rolling average of the last week" -- each row is treated independently
by most regression/tree algorithms. If the underlying process has memory
(today's chlorophyll is correlated with yesterday's wind, or with a
multi-day upwelling trend), that memory has to be exposed explicitly as
extra columns:

- **Lags** -- the value of a predictor N days earlier (e.g.
  `wind_u_ms_lag3`, `mur_sst_degC` shifted by a fixed offset).
- **Rolling summaries** -- a moving average or sum over a trailing window
  (e.g. `plv_solar_roll7d_wm2`, `cmems_upwelling_cumul14d_ms_d`).
- **Anomalies** -- a predictor's deviation from its typical seasonal value
  (e.g. `mur_sst_anom_doy_degC`, `chl_anom_log10_monthly`), which separates
  an unusually warm/cool day from the normal seasonal cycle.
- **Availability flags** -- a boolean column (e.g. `wind_available`,
  `chl_cons_available`) recording whether the underlying source actually
  had data that day, since satellite products have their own gaps
  (cloud cover, swath coverage) independent of the in-situ sensor's gaps.

Each lag/rolling/anomaly variant is a design decision, not something a
plain tabular model infers automatically. The curated feature table in
this repository already has many of these variants pre-computed; building
an equivalent table from scratch for a new sensor or site requires
re-doing this work (see `docs/methodology/model_families.md` and
`notebooks/09_adapting_the_workflow_to_a_new_sensor.ipynb`).

## 3. Loading the curated external-predictor feature table

`chlorophyll_predictor_features_curated.csv` has one row per calendar day
(3,988 rows) and 126 columns. Column families include:

- calendar: `season`, `day_of_year`, `month`, `year`, `doy_sin`, `doy_cos`
- satellite chlorophyll proxy: `chl_cons_log10`, `chl_perm_log10`, and
  their lag/roll/anomaly/patchiness variants
- sea-surface temperature: `mur_sst_degC`, `ostia_sst_degC`,
  `sst_primary_degC`, and SST gradient/frontal features
- wind: `wind_u_ms`, `wind_v_ms`, `wind_spd_ms` (CMEMS reanalysis) and
  `plv_wind_*` (nearby meteorological station)
- meteorological forcing: `plv_temp_degC`, `plv_pressure_hPa`,
  `plv_humid_pct`, `plv_precip_daily_mm`, `plv_solar_wm2`
- upwelling indices: `plv_upwelling_ms`, `cmems_upwelling_ms`, and their
  cumulative/relaxation-index variants

See `docs/data_dictionary.md` for the full column-by-column listing.

In [1]:
import pandas as pd

features = pd.read_csv(
    "../data_public/chlorophyll/chlorophyll_predictor_features_curated.csv",
    parse_dates=["date"],
)
print(features.shape)
features[[
    "date", "season", "chl_cons_log10", "mur_sst_degC",
    "wind_spd_ms", "plv_upwelling_ms",
]].head()

(3988, 126)


,date,season,chl_cons_log10,mur_sst_degC,wind_spd_ms,plv_upwelling_ms
0,2015-07-01,JJA,0.408127,13.270990,2.503007,-2.335232
1,2015-07-02,JJA,0.114188,13.269983,3.072794,-3.103372
2,2015-07-03,JJA,0.893770,13.260980,3.931441,-2.730098
3,2015-07-04,JJA,0.001893,13.451990,1.969993,-0.082689
4,2015-07-05,JJA,NaN,13.463983,2.624734,-1.892782


## 4. Why external predictors alone did not clearly beat interpolation here

In this benchmark's artificial-gap validation, external-predictor tabular
models (trained only on the feature families above, with no access to the
target's own recent history) did **not** show a clear, statistically
significant improvement over linear interpolation across most gap
lengths. Two probabilistic sequence models (a Gaussian process and a
state-space/Kalman model, see `docs/methodology/model_families.md`) and a
zero-shot foundation model (TS-ICL, see
`notebooks/06_tsicl_zero_shot_imputation.ipynb`) performed competitively
or better. The validated comparison numbers are in
`results_public/chlorophyll/chlorophyll_benchmark_summary.csv` and
`results_public/chlorophyll/chlorophyll_artificial_gap_scores.csv`.

Plausible reasons, in this local, relatively low-data setting (roughly a
decade of daily data at a single station):

- the record is short enough that a tabular model has limited examples to
  learn from, especially for less common conditions (long gaps, event
  days);
- external predictors capture broad physical forcing (temperature, wind,
  upwelling) but do not directly observe the target variable's own
  short-term persistence, which linear interpolation exploits by
  construction over short gaps;
- chlorophyll-a at this site is noisy and event-driven (see
  `docs/methodology/event_limitation.md`), and external predictors alone
  do not fully explain that variability.

More feature engineering and a more complex model family are not
guaranteed to beat a much simpler baseline; that comparison should always
be checked against validation-grade evidence (see
`docs/evidence_hierarchy.md`) rather than assumed.

## 5. The real canonical model: ExtraTrees on the 47-column `arm4` feature set

`experiments/chlorophyll/tabular_models.py` is the published, authoritative
external-tabular driver -- ported from the private project's Sprint-6H
lineage, the source of the "External tabular (ExtraTrees)" row in
`results_public/chlorophyll/chlorophyll_benchmark_summary.csv` and
`chlorophyll_matched_support_method_metrics.csv`. Its feature set
(`tabular_models.ARM4_COLUMNS`, 47 columns as published, 46 numeric after
`sst_primary_source` is dropped at fit time) is an exact, tested registry,
not an approximate description -- see `tests/test_tabular_models.py`.
`ExtraTreesRegressor(n_estimators=500)` is the **canonical** learner;
`HistGradientBoostingRegressor` is retained as a **diagnostic comparator
only** and must never be presented as the canonical result.

The cell below runs the real driver, leave-one-gap-out, on a small sample
of gaps (not the full 449-gap matched support -- that takes several
minutes and belongs in a background run, not an ordinary notebook
execution). To reproduce the full released comparison:

```bash
python -m experiments.chlorophyll.run_classical_benchmark \
    --methods ext_tabular_extratrees,ext_tabular_hgb \
    --out build/chlorophyll/classical_benchmark
python -m experiments.chlorophyll.run_classical_benchmark --verify \
    --out build/chlorophyll/classical_benchmark
```

which writes per-method predictions and a `summary_metrics.csv`, then
`--verify` compares that summary against the frozen released table
(`results_public/chlorophyll/chlorophyll_matched_support_method_metrics.csv`)
and classifies each method's reproduction as bit-identical / numerically
exact / within tolerance / not reproduced.

In [2]:
import sys

import numpy as np

sys.path.insert(0, "..")

from experiments.chlorophyll import benchmark_contract as bc
from experiments.chlorophyll import tabular_models as tm

target = pd.read_csv(
    "../data_public/chlorophyll/chlorophyll_daily_target.csv", parse_dates=["date"]
).set_index("date").sort_index()

arm4_cols = tm.load_arm4_numeric_columns(features.set_index("date"))
print(f"arm4: {len(arm4_cols)} numeric columns (of {len(tm.ARM4_COLUMNS)} published)")
print(f"forbidden target-history columns present: {tm.forbidden_target_history_columns(arm4_cols)}")

# A small, real sample from the released matched-support pool (see
# benchmark_contract.py) -- not the full 449 gaps, for notebook runtime.
sample = bc.load_matched_support_pool().sample(n=8, random_state=0)
preds_et, warns_et = tm.run_loco_evaluation(
    "ext_tabular_extratrees", sample, target, features.set_index("date"), arm4_cols
)
preds_hgb, warns_hgb = tm.run_loco_evaluation(
    "ext_tabular_hgb", sample, target, features.set_index("date"), arm4_cols
)

for name, preds in [("ExtraTrees (canonical)", preds_et), ("HGB (diagnostic)", preds_hgb)]:
    err = (preds["pred_log10"] - np.log10(preds["true"].clip(lower=1e-4))).abs()
    print(f"{name}: {len(preds)} predicted days, day-weighted MAE (log10) = {err.mean():.4f}")

print(
    "\nThis 8-gap sample's MAE is illustrative only -- it will not match the "
    "released 449-gap matched-support MAE in "
    "results_public/chlorophyll/chlorophyll_matched_support_method_metrics.csv "
    "(different, much smaller support). Run the CLI command above for the full comparison."
)

arm4: 46 numeric columns (of 47 published)
forbidden target-history columns present: []


ExtraTrees (canonical): 57 predicted days, day-weighted MAE (log10) = 0.1698
HGB (diagnostic): 57 predicted days, day-weighted MAE (log10) = 0.1768

This 8-gap sample's MAE is illustrative only -- it will not match the released 449-gap matched-support MAE in results_public/chlorophyll/chlorophyll_matched_support_method_metrics.csv (different, much smaller support). Run the CLI command above for the full comparison.


## 6. Takeaways

- External-predictor tabular models are useful because they generalize to
  any gap length and any position in the record, with no dependence on
  recent target observations.
- Building a good feature table requires designing lags, rolling windows,
  anomalies, and availability flags one at a time, ideally tracked in a
  feature registry with an explicit ablation plan rather than added all at
  once.
- In this project's low-data, single-station setting, external predictors
  alone did not clearly outperform linear interpolation under
  validation-grade testing -- see
  `results_public/chlorophyll/chlorophyll_benchmark_summary.csv` for the
  numbers, and `docs/evidence_hierarchy.md` before drawing conclusions
  from any other table in this repository.
- See `05_gap_edge_residual_models.ipynb` for a complementary model family
  that additionally uses gap-edge information, and
  `06_tsicl_zero_shot_imputation.ipynb` for the leading method in this
  benchmark under artificial-gap validation.